In [4]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import text_extraction, create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

/Users/hendrikweichel/miniconda3/envs/nace_project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [5]:
# Parameters: 

threshold_min_chunk_len = 100
cos_threshold = 0.4
sentence_length = 6

In [6]:
dataset_path = "../data/german_annual_reports"
dataset_path = "../data/stoxx_600_extended"
dataset_path = "../data/stoxx_600"

In [14]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [15]:
nace_classes = pd.read_excel(os.path.join(dataset_path, "STOXX600_extended.xlsx"))
nace_classes.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/stoxx_600/STOXX600_extended.xlsx'

In [16]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

NameError: name 'nace_classes' is not defined

In [17]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

NameError: name 'report_to_nace_class' is not defined

In [18]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/stoxx_600/TXTs/Hannover Rueck SE1.txt',
 '../data/stoxx_600/TXTs/Interpump Group S.p.A.1.txt',
 '../data/stoxx_600/TXTs/Intertek Group PLC1.txt',
 '../data/stoxx_600/TXTs/Anheuser-Busch InBev SANV3.txt',
 '../data/stoxx_600/TXTs/Scout24 SE3.txt',
 '../data/stoxx_600/TXTs/Bridgepoint Group Plc1.txt',
 '../data/stoxx_600/TXTs/Financiere de Tubize SA2.txt',
 '../data/stoxx_600/TXTs/Severn Trent Plc1.txt',
 '../data/stoxx_600/TXTs/Bakkafrost PF2.txt',
 '../data/stoxx_600/TXTs/Ferrari NV2.txt',
 '../data/stoxx_600/TXTs/Sika AG3.txt',
 '../data/stoxx_600/TXTs/Swiss Prime Site AG2.txt',
 '../data/stoxx_600/TXTs/Poste Italiane SpA2.txt',
 '../data/stoxx_600/TXTs/Haleon PLC1.txt',
 '../data/stoxx_600/TXTs/Land Securities Group PLC2.txt',
 '../data/stoxx_600/TXTs/Rentokil Initial plc2.txt',
 '../data/stoxx_600/TXTs/Nexans SA3.txt',
 '../data/stoxx_600/TXTs/SEB SA2.txt',
 '../data/stoxx_600/TXTs/Deutsche Bank Aktiengesellschaft1.txt',
 '../data/stoxx_600/TXTs/Technip Energies NV1.txt',


In [19]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [ ]:
for i in range(1): 
    nace_level = i

    result_path = f"../results/dataset__{dataset_name}_sentence_len_{sentence_length}__min_chunk_len_{threshold_min_chunk_len}__cos_thresh_{cos_threshold}__nace_level_{nace_level}"

    res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i)

  0%|          | 0/37 [00:00<?, ?it/s]

Report:  ../data/TEXT_stoxx600_extended_docling/Eurofins Scientific SE2.txt
Number of Chunks:  31


  3%|▎         | 1/37 [00:18<10:49, 18.04s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/TAG Immobilien AG3.txt
Number of Chunks:  1099


  5%|▌         | 2/37 [00:27<07:40, 13.15s/it]

...

 95%|█████████▍| 35/37 [80:23:06<07:14, 217.02s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/Phoenix Group Holdings plc1.txt
Number of Chunks:  2369


 97%|█████████▋| 36/37 [80:23:25<02:37, 157.59s/it]

Report:  ../data/TEXT_stoxx600_extended_docling/Bellway p.l.c.1.txt
Number of Chunks:  1372


100%|██████████| 37/37 [80:23:35<00:00, 7822.04s/it]


In [1]:
# test for different sentence lengths

# nace_level = 1
# for i in [3,4,5,6,7]: 

#     sentence_length = i

#     def preprocess_report(pdf_path: str) -> List[str]:

#         with open(pdf_path, "r") as f: 
#             text = f.read()
        
#         lines = text.split("\n")

#         # drop if condidtion is True
#         conditions = [
#             # filter images
#             lambda line: line == '<!-- image -->',
            
#             #filter tables 
#             lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

#             # filter headers
#             lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

#             # filter sentences
#             lambda line: "." not in line,
            
#             # more than 50% is numbers
#             lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

#             # minimum 3 words 
#             lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

#             # Minimum 2 Sentences
#             #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

#         ]
#         accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]

#         chunks = []

#         for line in accepted_lines: 
#             sentences = split_text_into_sentences(line, language='en')
#             sentences = [sentence.strip() for sentence in sentences]
#             sentences = [sentence for sentence in sentences if sentence != ""]
#             new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

#             chunks += new_chunks

#         # if there is only one sentence in the last chunk, balance the two last chunks
#         if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
#             last_two_chunks = chunks[-2] + " " + chunks[-1]
#             chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
#             chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

#         chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
#         chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
#         chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
#         chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
#         chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
#         chunks = [chunk.lower() for chunk in chunks]
#         chunks = [chunk.strip() for chunk in chunks]

#         return chunks
    
#     result_path = f"../results/paragraph_and_sentence_len_{sentence_length}_min_chunk_len_{threshold_min_chunk_len}_cos_thresh_{cos_threshold}_nace_level_{nace_level}_stoxx"

#     res = test_base.test_similarities(reports_path, preprocess_report, threshold_min_chunk_len, cos_threshold, report_to_nace_class, result_path, level=i)